In [1]:
import pandas as pd
import numpy as np


# Loading Dataset

In [2]:
df = pd.read_csv("IMDB_Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# Preprocessing

### Removing HTML Tags

In [4]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'',text)

In [5]:
df['review'] = df['review'].apply(remove_html_tags)

### Removing Punctuations

In [6]:
import string
exclude=string.punctuation #it will rteurn all the puctuations in python
def  remove_punctuation(text):
    for char in exclude:
        text = text.replace(char,'')
    return text

In [7]:
df['review'] = df['review'].apply(remove_punctuation)

### Slang Language Treatment

In [8]:
chat_word = {
    "U": "you",
    "R": "are",
    "UR": "your",
    "BRB": "be right back",
    "LOL": "laughing out loud",
    "OMG": "oh my god",
    "IDK": "I don’t know",
    "TBH": "to be honest",
    "BTW": "by the way",
    "GTG": "got to go",
    "BFF": "best friends forever",
    "ROFL": "rolling on the floor laughing",
    "TTYL": "talk to you later",
    "IMO": "in my opinion",
    "IMHO": "in my humble opinion",
    "FYI": "for your information",
    "LMAO": "laughing my ass off",
    "NP": "no problem",
    "THX": "thanks",
    "TY": "thank you",
    "PLZ": "please",
    "CUL8R": "see you later",
    "GR8": "great",
    "NVM": "never mind",
    "SMH": "shaking my head",
    "IDC": "I don’t care",
    "ILY": "I love you",
    "YOLO": "you only live once",
    "TMI": "too much information",
    "AFK": "away from keyboard",
    "OMW": "on my way",
    "FAQ": "frequently asked questions",
    "ASAP": "as soon as possible",
    "FOMO": "fear of missing out",
}
def chat_word_conversion(text):
    new_text = []
    for w in text.split(): # it will split the words in a sentence by spaces into a list.
        if w.upper() in chat_word: #Check if the word is in dictionary
            new_text.append(chat_word[w.upper()]) #This finds the full form of the chat word and appends it.
        else:
            new_text.append(w)
    return " ".join(new_text) #joins all words with spaces back into a sentence.

In [9]:
df['review'] = df['review'].apply(chat_word_conversion)

### Conversion to Lower Case


In [10]:
df['review'] = df['review'].str.lower()

### Emojis Handling

In [11]:
import emoji
def emoji_removal(text):
    text = emoji.demojize(text)
    return text

In [12]:
!pip install emoji


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [13]:
df['review'] = df['review'].apply(emoji_removal)

### Removal of Stop Words

In [14]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords = stopwords.words('english')
def remove_stopwords(text):
    new_text = []
    for word in text.split():
        if word in stopwords:
            new_text.append('')
        else:
            new_text.append(word)

    return " ".join(new_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
!pip install nltk



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [16]:
df['review'] = df['review'].apply(remove_stopwords)

### Tokenization

In [17]:
def tokenize(text):
  return text.split( )


In [18]:
df['review'] = df['review'].apply(tokenize)

### Lemitization

In [19]:
!python -m spacy download en_core_web_sm

C:\Users\DELL\AppData\Local\hermes\hermes-agent\venv\Scripts\python.exe: No module named spacy


In [20]:
import spacy
nlp = spacy.load('en_core_web_sm')
def token_lemmatized(text):
  text = str(text)
  doc = nlp(text)
  return " ".join([token.lemma_.lower() for token in doc if token.is_alpha])

In [ ]:
df['review'] = df['review'].apply(token_lemmatized)

### Vectorization

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
x = tfidf.fit_transform(df['review'])

### Label Encoding of OutputColumn

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])

### Train Test Split

In [14]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size = 0.2,random_state = 42)

### Model (Logistic Regression) Training

In [15]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(x_train,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### Prediction

In [16]:
y_pred = model.predict(x_test)

### Model Evaluation

## Accuracy

In [17]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test,y_pred)
print("Accuracy: ",accuracy)




Accuracy:  0.8997


## F1-Score

In [18]:
from sklearn.metrics import f1_score
f1score = f1_score(y_test,y_pred)
print("F1 Score: ",f1score)

F1 Score:  0.9015991366624154


## Precision

In [19]:
from sklearn.metrics import precision_score
precision = precision_score(y_test,y_pred)
print("Precision: ",precision)

Precision:  0.8915405510283275


## Recall

In [20]:
from sklearn.metrics import recall_score
recall = recall_score(y_test,y_pred)
print("Recall: ",recall)

Recall:  0.9118872792220679
